# Deep Dive into Apache Spark Cluster Architecture

**Module**: 01_spark_fundamentals  
**Notebook**: 02_spark_architecture.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: The deep internal architecture of Spark: Driver, Cluster Manager, Worker nodes, and Executors.
- **Why does this topic exist?**: To write scalable distributed code, you must understand where your code runs: on the driver thread or distributed across executor threads.
- **What problem does it solve?**: Developers often write driver-side bottlenecks (e.g. collecting massive datasets, serializing non-serializable objects) without realizing it.
- **Where is it used in real-world Data Engineering?**: Diagnosing production cluster crashes, OutOfMemory errors, and cluster sizing require architectural mastery.

---

## 2. Detailed Architecture Walkthrough

```
+-----------------------------------------------------------------------------------+
| DRIVER (Jupyter Container)                                                        |
|  - SparkContext & SparkSession                                                    |
|  - DAGScheduler (Creates Stages)                                                  |
|  - TaskScheduler (Dispatches Tasks)                                               |
|  - Web UI (Port 4040)                                                             |
+-------------------------+---------------------------------------------------------+
                          |
                          v RPC (spark://spark-master:7077)
+-----------------------------------------------------------------------------------+
| CLUSTER MANAGER (Standalone Master)                                               |
|  - Tracks Worker capacity (Cores & RAM)                                           |
|  - Allocates Executor slots to applications                                       |
+-------------------------+---------------------------------------------------------+
                          | Launches Executors
             +------------+------------+
             |                         |
             v                         v
+------------------------+  +------------------------+
| WORKER 1 (Port 8081)   |  | WORKER 2 (Port 8082)   |
|  Worker Daemon         |  |  Worker Daemon         |
|  +------------------+  |  |  +------------------+  |
|  | Executor JVM     |  |  |  | Executor JVM     |  |
|  | - 2 Cores        |  |  |  | - 2 Cores        |  |
|  | - 4 GB RAM       |  |  |  | - 4 GB RAM       |  |
|  | - BlockManager   |  |  |  | - BlockManager   |  |
|  | - Task Threads   |  |  |  | - Task Threads   |  |
|  +------------------+  |  |  +------------------+  |
+------------------------+  +------------------------+
```

### Driver vs Executor Code Execution:
- Code written outside transformations executes **once on the Driver**.
- Code inside `map`, `filter`, `foreachPartition` executes **concurrently inside Executor JVM task threads**.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Spark_Architecture_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Investigating the Driver & SparkContext Configuration

Let us inspect the live properties configured for this cluster application.

In [ ]:
conf = spark.sparkContext.getConf()
print("=== Key Cluster Configurations ===")
for key in [
    "spark.master",
    "spark.app.name",
    "spark.driver.host",
    "spark.driver.port",
    "spark.executor.memory",
    "spark.sql.shuffle.partitions",
    "spark.eventLog.enabled"
]:
    print(f"  {key:30}: {conf.get(key, 'NOT SET')}")

## 4. Tracing Driver vs Executor Execution Contexts

Watch what prints on the Driver vs what prints in Executor worker logs!

In [ ]:
import os

print(f" DRIVER is running on container: {os.uname().nodename}")

def log_worker_info(partition_index, iterator):
    worker_host = os.uname().nodename
    yield f"Task for partition {partition_index} executed on executor container: {worker_host}"

results = (
    spark.sparkContext
    .parallelize(range(10), numSlices=4)
    .mapPartitionsWithIndex(log_worker_info)
    .collect()
)

print("\n EXECUTOR execution locations:")
for r in results:
    print(f"  - {r}")

---

## Experiment: Non-Serializable Object Driver-to-Executor Passing

- **1. What are we testing?**: Observe what happens when passing an un-serializable Python object into an executor task.
- **2. What do we expect?**: Spark raises a PickleError / Py4J serialization error because task closures must be serialized over the network.

In [ ]:
import socket

# socket objects cannot be pickled or serialized across network
sock = socket.socket()

try:
    spark.sparkContext.parallelize([1, 2, 3]).map(lambda x: (x, str(sock))).collect()
    print("Execution succeeded.")
except Exception as e:
    print(f"Caught expected serialization error: {type(e).__name__}")
finally:
    sock.close()

**3. What happened?**: Spark threw an exception during task closure serialization.  
**4. Why did it happen?**: Every variable referenced inside a transformation lambda must be serialized via pickle to executor processes.  
**5. Practical Engineering Lesson**: Instantiate connection objects (DB connections, sockets, files) INSIDE `mapPartitions` on the executor, never on the driver.

---

## Practical Exercises

### Exercise 1 (Beginner): Inspect Active Application ID and Web UI URL
**Problem Statement**: Retrieve the applicationId and webUrl from sparkContext and print them.  
**Input / Context**: spark.sparkContext

In [ ]:
# Exercise 1: App ID and UI URL

### Exercise 2 (Intermediate): Count Cores Used Across Worker Nodes
**Problem Statement**: Run a job with 8 partitions and verify how many unique container hostnames contributed.  
**Input / Context**: sc.parallelize(range(80), 8)

In [ ]:
# Exercise 2: Count active worker nodes

### Exercise 3 (Advanced): Demonstrate mapPartitions DB Connection Pattern
**Problem Statement**: Use mapPartitions to simulate opening a database connection once per partition rather than once per row.  
**Input / Context**: RDD of 100 elements across 4 partitions

In [ ]:
# Exercise 3: mapPartitions connection pattern

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Inspect Active Application ID and Web UI URL):
```python
print("Application ID:", spark.sparkContext.applicationId)
print("Driver Web UI:", spark.sparkContext.uiWebUrl)
```

#### Solution 2 (Count Cores Used Across Worker Nodes):
```python
nodes = spark.sparkContext.parallelize(range(80), 8).map(lambda _: os.uname().nodename).distinct().collect()
print(f"Total distinct nodes executing work: {len(nodes)} -> {nodes}")
```

#### Solution 3 (Demonstrate mapPartitions DB Connection Pattern):
```python
def process_partition_with_mock_conn(iterator):
    # Opened once per partition on the executor JVM/Python worker
    conn = f"MockDBConn_{os.uname().nodename}"
    processed = []
    for item in iterator:
        processed.append(f"{item}_saved_via_{conn}")
    return iter(processed)

res = spark.sparkContext.parallelize(range(8), 2).mapPartitions(process_partition_with_mock_conn).collect()
print("Sample results from mapPartitions:", res[:4])
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Driver Role**: Coordinates DAG, breaks jobs into stages, dispatches tasks, hosts Web UI.
- **Executor Role**: Executes task threads, caches data blocks, serves shuffle data.
- **Serialization**: All code referenced in transformations must be serializable.

### Essential Technical Terms
- `Driver`: The orchestrating JVM process running the Spark application.
- `Executor`: Worker JVM process running task slots.
- `Closure`: Variables and functions packaged to be sent over network to executors.

### Common Engineering Mistakes
- **Initializing heavy resources on Driver**: Opening DB connections on driver and passing to lambdas fails with serialization errors.
- **Calling collect() on large outputs**: Causes Driver OOM.

### Interview Questions on This Topic
- **Q: What is the difference between a Worker and an Executor?**
  - *A*: A Worker is a long-lived machine daemon; an Executor is an application-specific JVM spawned by the worker.
- **Q: Where does the Spark UI run?**
  - *A*: Embedded inside the Driver process on port 4040.

### What to Study Next
Open 01_spark_fundamentals/03_spark_session_and_spark_context.ipynb to master session management.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')